In [1]:
import numpy as np
import pandas as pd

### Day 1 — Bird Diversity & Habitat Analysis

In [2]:
np.random.seed(42)

sites = ["East Kolkata Wetlands", "Sundarban Fringe", "Nalban", "Santragachi"]
species = ["Egret", "Kingfisher", "Heron", "Cormorant", "Ibis", "Pied Hornbill"]

dates = pd.date_range("2025-01-01", "2025-06-01", freq="MS")

rows = []

for date in dates:
    for site in sites:
        for sp in species:
            rows.append({
                "date": date,
                "site": site,
                "species": sp,
                "count": np.random.poisson(
                    np.random.uniform(5, 30)
                ),
                "observation_hours": np.random.uniform(2, 8)
            })

birds = pd.DataFrame(rows)

# Introduce some missing observations
missing_idx = np.random.choice(
    birds.index,
    size=12,
    replace=False
)

birds.loc[missing_idx, "count"] = np.nan

birds.head()

,date,site,species,count,observation_hours
0,2025-01-01,East Kolkata Wetlands,Egret,15.0,2.935967
1,2025-01-01,East Kolkata Wetlands,Kingfisher,6.0,3.090950
2,2025-01-01,East Kolkata Wetlands,Heron,9.0,3.198043
3,2025-01-01,East Kolkata Wetlands,Cormorant,19.0,5.645269
4,2025-01-01,East Kolkata Wetlands,Ibis,8.0,4.971061


In [ ]:
birds.info()
birds.describe()

In [ ]:
birds['count'].unique()
birds[birds['count'].isna()]
birds.fillna(birds['count'].mean(), inplace=True)

In [62]:
birds['month'] = birds['date'].dt.month
birds['day_name'] = birds['date'].dt.day_name()
birds['density'] = birds['count'] / birds['observation_hours']
birds.head()

,date,site,species,count,observation_hours,month,day_name,density
0,2025-01-01,East Kolkata Wetlands,Egret,15.0,2.935967,1,Wednesday,5.109049
1,2025-01-01,East Kolkata Wetlands,Kingfisher,6.0,3.090950,1,Wednesday,1.941151
2,2025-01-01,East Kolkata Wetlands,Heron,9.0,3.198043,1,Wednesday,2.814221
3,2025-01-01,East Kolkata Wetlands,Cormorant,19.0,5.645269,1,Wednesday,3.365650
4,2025-01-01,East Kolkata Wetlands,Ibis,8.0,4.971061,1,Wednesday,1.609314


In [5]:
birds[birds['count']>birds['count'].mean()]
birds.query('species == "Egret" | species =="Kingfisher" | species == "Heron" | species == "Cormorant" | species == "Ibis"').where(birds['count'] > birds['count'].mean()).dropna()
birds.query('species == "Kingfisher"').where(birds['site'] == 'Nalban').dropna().sort_values(by='date',ascending=False)

,date,site,species,count,observation_hours,month,day_name,density
133,2025-06-01,Nalban,Kingfisher,29.0,6.179905,6.0,Sunday,4.692629
109,2025-05-01,Nalban,Kingfisher,31.0,2.610695,5.0,Thursday,11.874233
85,2025-04-01,Nalban,Kingfisher,14.0,4.258780,4.0,Tuesday,3.287326
61,2025-03-01,Nalban,Kingfisher,3.0,7.069252,3.0,Saturday,0.424373
37,2025-02-01,Nalban,Kingfisher,26.0,5.945677,2.0,Saturday,4.372925
13,2025-01-01,Nalban,Kingfisher,16.0,7.323276,1.0,Wednesday,2.184814


In [ ]:
birds.where(birds['observation_hours']>=birds['observation_hours'].quantile(0.9)).dropna()
birds['observation_hours'].quantile([0.25,0.5,0.75,0.9])

In [69]:
birds.groupby(['site','species'])['count'].agg(['min','max','mean','sum']).sort_values(by=['site','sum'] ,ascending=[True,False] )
birds.groupby('species')['count'].agg(['max','min','std'])
birds.groupby(['site','month'])['count'].agg(['sum']).sort_values(by=['site','sum'],ascending=[True,False])

sum
site                  month            
East Kolkata Wetlands 5      105.280303
                      6      101.000000
                      3       91.000000
                      4       87.560606
                      2       83.280303
                      1       63.000000
Nalban                6      161.000000
                      4      126.000000
                      5      105.000000
                      2      102.280303
                      3       98.280303
                      1       84.280303
Santragachi           4      130.280303
                      2      129.280303
                      5      108.000000
                      3       92.280303
                      1       92.000000
                      6       68.000000
Sundarban Fringe      1      134.000000
                      2      111.000000
                      3      111.000000
                      6      104.280303
                      4      101.280303
                      5       99.000000

In [66]:
birds_pivot = pd.pivot_table(birds, index='site', columns='species',values='count',aggfunc='sum')
birds_pivot

species,Cormorant,Egret,Heron,Ibis,Kingfisher,Pied Hornbill
site,,,,,,
East Kolkata Wetlands,127.560606,68.000000,76.280303,87.000000,62.280303,110.000000
Nalban,119.280303,95.000000,101.280303,123.000000,119.000000,119.280303
Santragachi,98.000000,85.280303,76.280303,175.280303,90.000000,95.000000
Sundarban Fringe,83.000000,111.280303,116.000000,94.000000,110.000000,146.280303


In [68]:
ovser_pivot = pd.pivot_table(birds,index='site',columns='species',values='observation_hours',aggfunc='mean')
ovser_pivot

species,Cormorant,Egret,Heron,Ibis,Kingfisher,Pied Hornbill
site,,,,,,
East Kolkata Wetlands,4.937580,5.210786,4.481279,3.789049,4.921066,6.052213
Nalban,6.111787,4.531874,5.274141,5.247168,5.564598,5.354473
Santragachi,6.066084,4.617324,4.717663,4.939191,5.527176,5.546335
Sundarban Fringe,4.175224,5.312795,5.317632,3.914753,4.800563,6.294670
